# FLUX.1-dev LoRA Fine-Tuning on RunPod RTX 4090 (24GB VRAM)
This notebook provides a complete, ready-to-run pipeline for training a FLUX.1-dev LoRA using the Ostris AI-Toolkit. It is specifically optimized for a single 24GB VRAM GPU (e.g., RTX 4090).

## 1. Environment Setup
Clone the Ostris AI-Toolkit repository and install all required dependencies.

In [ ]:
!git clone https://github.com/ostris/ai-toolkit.git
%cd ai-toolkit
!git submodule update --init --recursive
!pip install -r requirements.txt
!pip install huggingface_hub

### Authenticate with Hugging Face
FLUX.1-dev is a gated model. You must provide a Hugging Face User Access Token (with read access) to download the base model.

In [ ]:
from huggingface_hub import login

# Replace 'YOUR_HF_TOKEN' with your actual Hugging Face token
hf_token = 'YOUR_HF_TOKEN'
login(token=hf_token)

## 2. Dataset Directory Setup
Create the directories for the dataset and output, and verify that the images and text files are correctly paired.

In [ ]:
import os

dataset_dir = '/workspace/dataset'
output_dir = '/workspace/output'

os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print(f'Created dataset directory at: {dataset_dir}')
print(f'Created output directory at: {output_dir}')

### Dataset Verification
Run this script to ensure your `.jpg`, `.jpeg`, and `.png` files have matching `.txt` caption files.

In [ ]:
import os

dataset_dir = '/workspace/dataset'

# Valid image extensions
img_exts = {'.jpg', '.jpeg', '.png'}

if os.path.exists(dataset_dir):
    files = os.listdir(dataset_dir)
    images = [f for f in files if os.path.splitext(f)[1].lower() in img_exts]
    captions = [f for f in files if os.path.splitext(f)[1].lower() == '.txt']

    image_basenames = {os.path.splitext(f)[0] for f in images}
    caption_basenames = {os.path.splitext(f)[0] for f in captions}

    unmatched_images = image_basenames - caption_basenames
    unmatched_captions = caption_basenames - image_basenames

    print(f'Found {len(images)} images and {len(captions)} captions.')

    if not unmatched_images and not unmatched_captions:
        print('All images and captions are perfectly paired!')
    else:
        if unmatched_images:
            print(f'Warning: Missing captions for images: {unmatched_images}')
        if unmatched_captions:
            print(f'Warning: Missing images for captions: {unmatched_captions}')
else:
    print(f'Dataset directory {dataset_dir} does not exist yet.')

## 3. YAML Configuration Generation
Dynamically generate the `job.yaml` configuration file required by AI-Toolkit. This configuration uses 8-bit Adam, bf16 precision, and fp8 base model quantization to fit within 24GB VRAM.

In [ ]:
import yaml
import os

# We should be in the ai-toolkit directory
config_path = 'config/job.yaml'

config = {
    'job': 'extension',
    'config': {
        'name': 'flux_lora_runpod',
        'process': [{
            'type': 'sd_trainer',
            'training_folder': '/workspace/output',
            'device': 'cuda:0',
            'trigger_word': 'vga_founder',
            'network': {
                'type': 'lora',
                'linear': 16,
                'linear_alpha': 16
            },
            'save': {
                'dtype': 'bfloat16',
                'save_every': 500,
                'max_step_saves_to_keep': 4,
                'push_to_hub': False
            },
            'datasets': [{
                'folder_path': '/workspace/dataset',
                'caption_ext': 'txt',
                'caption_dropout_rate': 0.05,
                'shuffle_tokens': False,
                'cache_latents_to_disk': True,
                'resolution': [512, 768, 1024]
            }],
            'train': {
                'batch_size': 1,
                'steps': 1500,
                'gradient_accumulation_steps': 1,
                'train_unet': True,
                'train_text_encoder': False,
                'gradient_checkpointing': True,
                'noise_scheduler': 'flowmatch',
                'optimizer': 'adamw8bit',
                'lr': 4e-4,
                'ema_config': {
                    'use_ema': True,
                    'ema_decay': 0.99
                },
                'dtype': 'bf16'
            },
            'model': {
                'name_or_path': 'black-forest-labs/FLUX.1-dev',
                'is_flux': True,
                'quantize': True  # Required for 24GB VRAM
            },
            'sample': {
                'sampler': 'flowmatch',
                'sample_every': 500,
                'sample_start_step': 0,
                'width': 1024,
                'height': 1024,
                'prompts': [
                    '[trigger] holding a sign that says "I LOVE PROMPTS!"',
                    '[trigger] drinking coffee at a cafe'
                ],
                'neg': '',
                'seed': 42,
                'walk_seed': True,
                'guidance_scale': 4,
                'sample_steps': 20
            }
        }],
        'meta': {
            'name': '[name]',
            'version': '1.0'
        }
    }
}

os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, sort_keys=False)

print(f'Successfully generated {config_path}')


## 4. Training Execution
Launch the training script. Ensure you are inside the `ai-toolkit` directory. The process may take a while depending on the number of steps and dataset size.

In [ ]:
!python run.py config/job.yaml

## 5. Export & Cleanup
Locate the final `.safetensors` LoRA model in the output directory and package it into a zip file for easy downloading.

In [ ]:
import os
import zipfile
import glob

output_dir = '/workspace/output/flux_lora_runpod'
zip_filename = '/workspace/output/flux_lora_final.zip'

# Find the safetensors file with the highest step count or the final one
safetensors_files = glob.glob(os.path.join(output_dir, '**', '*.safetensors'), recursive=True)

if not safetensors_files:
    print('No .safetensors files found in the output directory.')
else:
    # Sort files by modification time or step number (assuming standard naming conventions)
    safetensors_files.sort(key=os.path.getmtime)
    final_model = safetensors_files[-1]
    
    print(f'Found final model: {final_model}')
    
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(final_model, os.path.basename(final_model))
        
    print(f'Successfully zipped final model to: {zip_filename}')
